#### Criando Database

In [0]:
spark.sql("CREATE DATABASE IF NOT EXISTS gold")

DataFrame[]

#### 1o Projeto: Visão Comercial e Volume de Produtos

##### Tabela Principal (gold.fat_vendas_comercial):

In [0]:
from pyspark.sql.functions import (year, month, countDistinct, count, sum as spark_sum, round as spark_round, col)

#lendo as tabelas silver:
itens = spark.table("silver.fat_itens_pedidos")
pedidos = spark.table("silver.fat_pedidos").select("id_pedido", "data_pedido")
produtos = spark.table("silver.dim_produtos").select("id_produto", "categoria_produto")
trad = spark.table("silver.dim_categoria_produtos_traducao").select(
    col("nome_produto_pt").alias("categoria_produto"),
    col("nome_produto_en").alias("categoria_produto_en"))
total = spark.table("silver.fat_pedido_total").select(
    "id_pedido", "valor_total_pago_brl", "valor_total_pago_usd")




#join das tabelas (dataframe enriquecido)
df = itens.join(pedidos, on="id_pedido", how="left") \
         .join(produtos, on="id_produto", how="left") \
         .join(trad, on="categoria_produto", how="left") \
         .join(total, on="id_pedido", how="left")


#colunas derivadas
df = df.withColumn("ano_venda", year(col("data_pedido")))
df = df.withColumn("mes_venda", month(col("data_pedido")))

#agragando por ano, mes e categoria 
#e calculando metricas
df_agg = df.groupBy("ano_venda", "mes_venda", "categoria_produto").agg(
    countDistinct("id_pedido").alias("total_pedidos"),
    count("id_item").alias("qtd_itens_vendidos"),
    spark_round(spark_sum("preco_BRL"), 2).alias("receita_total_brl"),
    spark_round(spark_sum("valor_total_pago_usd"), 2).alias("receita_total_usd"),
    spark_round(spark_sum("preco_BRL") / countDistinct("id_pedido"), 2).alias("ticket_medio_brl")
)


df_agg.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable("gold.fat_vendas_comercial")
print("gold.fat_vendas_comercial criado!")





gold.fat_vendas_comercial criado!


##### Rankings Comerciais (Exibição via display()):

In [0]:
#top 5 produtos mais vendidos
top5_mais = df.groupBy("id_produto", "categoria_produto") \
    .agg(count("id_item").alias("qtd_vendida")) \
    .orderBy(col("qtd_vendida").desc()) \
    .limit(5)
display(top5_mais)


#top 5 produtos menos vendidos
top5_menos = df.groupBy("id_produto", "categoria_produto") \
    .agg(count("id_item").alias("qtd_vendida")) \
    .orderBy(col("qtd_vendida").asc()) \
    .limit(5)
display(top5_menos)


id_produto,categoria_produto,qtd_vendida
aca2eb7d00ea1a7b8ebd4e68314663af,moveis_decoracao,527
99a4788cb24856965c36a24e339b6058,cama_mesa_banho,488
422879e10f46682990de24d770e7f83d,ferramentas_jardim,484
389d119b48cf3043d311335e499d9c6b,ferramentas_jardim,392
368c6c730842d78016ad823897a372db,ferramentas_jardim,388


id_produto,categoria_produto,qtd_vendida
cb3ad2d2499eebaa7ccb4fd1e7db9bc7,automotivo,1
fa11ecd35f999783e96ac500532d9d45,cool_stuff,1
41eee23c25f7a574dfaf8d5c151dbb12,null,1
cdb8d3c880b6639a70764c6734e6bb69,beleza_saude,1
2fc7741cb6ea76e7164f448adc44652d,cama_mesa_banho,1


#### 2o Projeto: Satisfação de Clientes e Qualidade de Parceiros

##### Tabela Principal (gold.fat_vendas_comercial):

In [0]:
from pyspark.sql.functions import avg, sum as spark_sum, count, when, round as spark_round

#leitura da silver
aval = spark.table("silver.fat_avaliacoes_pedidos")
itens = spark.table("silver.fat_itens_pedidos")
produtos = spark.table("silver.dim_produtos").select("id_produto", "categoria_produto")
vendedores = spark.table("silver.dim_vendedores")


#join
df2 = aval.join(itens, on="id_pedido", how="left") \
          .join(produtos, on="id_produto", how="left") \
          .join(vendedores, on="id_vendedor", how="left")

# Cast explícito para garantir que nota_avaliacao é numérica
df2 = df2.withColumn("nota_avaliacao", col("nota_avaliacao").cast("integer"))

#agrupamento por categoria, nome do vendedor e estado do vendedor
#calculo de metricas
df2_agg = df2.groupBy("categoria_produto", "id_vendedor", "estado").agg(
    count("id_avaliacao").alias("total_avaliacoes"),
    spark_round(avg("nota_avaliacao"), 2).alias("avaliacao_media"),
    spark_sum(when(col("nota_avaliacao") >= 4, 1).otherwise(0)).alias("total_avaliacoes_positivas"),
    spark_sum(when(col("nota_avaliacao") <= 2, 1).otherwise(0)).alias("total_avaliacoes_negativas"),
    spark_round(
        spark_sum(when(col("nota_avaliacao") >= 4, 1).otherwise(0)) / count("id_avaliacao") * 100, 2
    ).alias("percentual_satisfacao")
)


df2_agg.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable("gold.fat_avaliacoes_clientes")
print("gold.fat_avaliacoes_clientes criado!")


gold.fat_avaliacoes_clientes criado!


##### Rankings de Qualidade (Exibição via display()):

In [0]:
# Por produto
prod_rank = df2.groupBy("id_produto", "categoria_produto") \
    .agg(spark_round(avg("nota_avaliacao"),2).alias("media"), count("id_avaliacao").alias("total"))


display(prod_rank.orderBy(col("media").desc(), col("total").desc()).limit(1))  # Mais bem avaliado
display(prod_rank.orderBy(col("media").asc(), col("total").desc()).limit(1))   # Menos bem avaliado


# Por vendedor
vend_rank = df2.groupBy("id_vendedor", "estado") \
    .agg(spark_round(avg("nota_avaliacao"),2).alias("media"), count("id_avaliacao").alias("total"))


display(vend_rank.orderBy(col("media").desc(), col("total").desc()).limit(1))  # Mais bem avaliado
display(vend_rank.orderBy(col("media").asc(), col("total").desc()).limit(1))   # Menos bem avaliado


id_produto,categoria_produto,media,total
37eb69aca8718e843d897aa7b82f462d,ferramentas_jardim,5.0,15


id_produto,categoria_produto,media,total
fb29f48bfea41db52e349454f433340e,informatica_acessorios,1.0,10


id_vendedor,estado,media,total
48efc9d94a9834137efd9ea76b065a38,PR,5.0,34


id_vendedor,estado,media,total
8d92f3ea807b89465643c219455e7369,SP,1.0,8
